In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import seaborn as sns

train = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/interactions_train.csv')
n_users = train.user_id.nunique()
n_items = train.recipe_id.nunique()
n_ratings = len(train)
rating_matrix_size = n_users * n_items
sparsity = 1 - n_ratings / rating_matrix_size

print(f"SPARSITY: {sparsity * 100.0:.2f}%")


In [ ]:
user_stats = train.groupby('user_id')['rating'].agg(
    mean_rating='mean',
    rating_count='count',
    std_rating='std'
).reset_index()


# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution of Per-User Rating Statistics', fontsize=14)

# Mean rating distribution
sns.histplot(user_stats['mean_rating'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(user_stats['mean_rating'].mean(), color='red', linestyle='--', label=f"Overall mean: {user_stats['mean_rating'].mean():.2f}")
axes[0].set_title('Distribution of Mean Rating per User')
axes[0].set_xlabel('Mean Rating')
axes[0].set_ylabel('Number of Users')
axes[0].legend()

# Rating count distribution (log scale because it's usually heavily skewed)
sns.histplot(user_stats['rating_count'], bins=50, kde=False, ax=axes[1], color='coral')
axes[1].set_yscale('log')
axes[1].set_title('Distribution of Rating Count per User')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Users (log scale)')

# Std rating distribution
sns.histplot(user_stats['std_rating'].dropna(), bins=30, kde=True, ax=axes[2], color='mediumseagreen')
axes[2].axvline(user_stats['std_rating'].mean(), color='red', linestyle='--', label=f"Overall mean std: {user_stats['std_rating'].mean():.2f}")
axes[2].set_title('Distribution of Rating Std Dev per User')
axes[2].set_xlabel('Std Dev of Ratings')
axes[2].set_ylabel('Number of Users')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Look at percentiles to find natural boundaries
percentiles = [10, 25, 50, 75, 90, 95, 99]
print(user_stats['rating_count'].describe())
print("\nPercentiles:")
for p in percentiles:
    print(f"  {p}th percentile: {np.percentile(user_stats['rating_count'], p):.0f} ratings")

In [ ]:
# Adjust thresholds after looking at your percentiles
COLD_THRESHOLD = 5
HOT_THRESHOLD = 20

def classify_user(count):
    if count < COLD_THRESHOLD:
        return 'cold'
    elif count < HOT_THRESHOLD:
        return 'warm'
    else:
        return 'hot'

user_stats['user_type'] = user_stats['rating_count'].apply(classify_user)

# Plot mean rating distribution split by user type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Mean Rating Distribution by User Type', fontsize=14)

colors = {'cold': 'coral', 'warm': 'steelblue', 'hot': 'mediumseagreen'}

for ax, user_type in zip(axes, ['cold', 'warm', 'hot']):
    subset = user_stats[user_stats['user_type'] == user_type]
    sns.histplot(subset['mean_rating'], bins=20, kde=True, ax=ax, color=colors[user_type])
    ax.set_title(f"{user_type.capitalize()} Users (n={len(subset):,})")
    ax.set_xlabel('Mean Rating')
    ax.set_ylabel('Number of Users')
    ax.axvline(subset['mean_rating'].mean(), color='black', linestyle='--',
               label=f"Mean: {subset['mean_rating'].mean():.2f}")
    ax.legend()

plt.tight_layout()
plt.show()

# Summary table
print(user_stats.groupby('user_type')[['mean_rating', 'rating_count', 'std_rating']].describe())

In [ ]:
COLD_THRESHOLD = 5
HOT_THRESHOLD = 20

user_stats['user_type'] = pd.cut(
    user_stats['rating_count'],
    bins=[0, COLD_THRESHOLD, HOT_THRESHOLD, float('inf')],
    labels=['cold', 'warm', 'hot']
)

print(user_stats['user_type'].value_counts())
print(user_stats['user_type'].value_counts(normalize=True).mul(100).round(1))

cold (46.5%)  → content-based only
               + global popularity fallback
               + any available context (time of day, cuisine filters)

warm (33.3%)  → weighted hybrid
               (lean content-based, sprinkle collaborative)
               weight = f(rating_count)

hot (20.2%)   → collaborative filtering primary
               + content-based for new recipe discovery

In [ ]:
recipes_df = pd.read_csv('/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_recipes.csv')

In [ ]:
train.columns

In [ ]:
recipes_df.columns

In [ ]:
# Check coverage between interaction and recipe data
interaction_recipes = set(train['recipe_id'].unique())
content_recipes = set(recipes_df['id'].unique())

print(f"Recipes in interactions: {len(interaction_recipes):,}")
print(f"Recipes in content data: {len(content_recipes):,}")
print(f"Interactions with no content record: {len(interaction_recipes - content_recipes):,}")
print(f"Content records never interacted with: {len(content_recipes - interaction_recipes):,}")

In [ ]:
# Isolate never-interacted recipes
cold_recipes = recipes_df[~recipes_df['id'].isin(interaction_recipes)]

# Compare token sequence lengths between cold and interacted recipes
interacted_recipes = recipes_df[recipes_df['id'].isin(interaction_recipes)]

print("=== Interacted Recipes ===")
print(f"Avg name tokens: {interacted_recipes['name_tokens'].apply(len).mean():.1f}")
print(f"Avg ingredient tokens: {interacted_recipes['ingredient_tokens'].apply(len).mean():.1f}")
print(f"Avg steps tokens: {interacted_recipes['steps_tokens'].apply(len).mean():.1f}")

print("\n=== Never-Interacted Recipes ===")
print(f"Avg name tokens: {cold_recipes['name_tokens'].apply(len).mean():.1f}")
print(f"Avg ingredient tokens: {cold_recipes['ingredient_tokens'].apply(len).mean():.1f}")
print(f"Avg steps tokens: {cold_recipes['steps_tokens'].apply(len).mean():.1f}")

# Check calorie level distribution
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Calorie Level Distribution: Interacted vs Never-Interacted Recipes', fontsize=13)

sns.countplot(x='calorie_level', data=interacted_recipes, ax=axes[0], color='steelblue')
axes[0].set_title(f'Interacted Recipes (n={len(interacted_recipes):,})')
axes[0].set_xlabel('Calorie Level')

sns.countplot(x='calorie_level', data=cold_recipes, ax=axes[1], color='coral')
axes[1].set_title(f'Never-Interacted Recipes (n={len(cold_recipes):,})')
axes[1].set_xlabel('Calorie Level')

plt.tight_layout()
plt.show()

In [ ]:
interacted_recipes.calorie_level.value_counts()

In [ ]:
cold_recipes.calorie_level.value_counts()

In [ ]:
# Compute per-recipe interaction stats
recipe_stats = train.groupby('recipe_id')['rating'].agg(
    mean_rating='mean',
    rating_count='count',
    std_rating='std'
).reset_index()

print(recipe_stats['rating_count'].describe())
print("\nPercentiles:")
for p in percentiles:
    print(f"  {p}th percentile: {np.percentile(recipe_stats['rating_count'], p):.0f} ratings")

In [ ]:
# How much of total interactions do the top recipes account for?
recipe_stats_sorted = recipe_stats.sort_values('rating_count', ascending=False)
total_interactions = recipe_stats_sorted['rating_count'].sum()

for pct in [0.01, 0.05, 0.10]:
    n_recipes = int(len(recipe_stats_sorted) * pct)
    top_interactions = recipe_stats_sorted.head(n_recipes)['rating_count'].sum()
    print(f"Top {pct*100:.0f}% recipes ({n_recipes:,} recipes) "
          f"account for {top_interactions/total_interactions*100:.1f}% of interactions")

# Plot the interaction distribution (log scale)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Recipe Interaction Distribution', fontsize=13)

# Raw distribution
sns.histplot(recipe_stats['rating_count'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Raw Distribution (heavily skewed)')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Number of Recipes')

# Log scale to see the tail
sns.histplot(np.log1p(recipe_stats['rating_count']), bins=50, ax=axes[1], color='coral')
axes[1].set_title('Log-Transformed Distribution')
axes[1].set_xlabel('log(1 + Number of Ratings)')
axes[1].set_ylabel('Number of Recipes')

plt.tight_layout()
plt.show()

Top 1%  (1,609 recipes)  → 21.3% of interactions
Top 5%  (8,045 recipes)  → 40.4% of interactions  
Top 10% (16,090 recipes) → 52.1% of interactions
Bottom 90% (144,811 recipes) → 47.9% of interactions


USERS
├── 25,076 total users
├── 46.5% cold (<5 ratings)
├── 33.3% warm (5-20 ratings)
└── 20.2% hot (>20 ratings)

RECIPES  
├── 178,265 total recipes
├── 160,901 have at least 1 interaction
├── 17,364 never interacted (but content quality is fine)
└── Median of only 2 interactions per recipe

INTERACTIONS
├── Ratings heavily skewed above 5 (80%+)
├── Top 10% recipes account for 52.1% of interactions
└── Popularity bias present but manageable

In [ ]:
raw_recipes = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_recipes.csv')
raw_recipes.columns

In [ ]:
# Get a quick overview of the additional fields
print("=== Sample rows ===")
print(raw_recipes[['minutes', 'tags', 'nutrition', 'n_steps', 'n_ingredients']].head(3))

print("\n=== Null counts ===")
print(raw_recipes[['minutes', 'tags', 'nutrition', 'n_steps', 'n_ingredients']].isnull().sum())

print("\n=== Minutes distribution ===")
print(raw_recipes['minutes'].describe())

print("\n=== n_steps and n_ingredients ===")
print(raw_recipes[['n_steps', 'n_ingredients']].describe())

print("\n=== Sample nutrition entry ===")
print(raw_recipes['nutrition'].head(3))

print("\n=== Sample tags entry ===")
print(raw_recipes['tags'].head(3))

In [ ]:
import ast

# Check minutes outliers
print("=== Minutes percentiles (high end) ===")
for p in [90, 95, 99, 99.9]:
    print(f"  {p}th percentile: {np.percentile(raw_recipes['minutes'], p):.0f} mins")

print(f"\nRecipes with minutes > 1440 (24 hours): {(raw_recipes['minutes'] > 1440).sum():,}")
print(f"Recipes with minutes == 0: {(raw_recipes['minutes'] == 0).sum():,}")

# Parse tags and check unique tag count
raw_recipes['tags_parsed'] = raw_recipes['tags'].apply(ast.literal_eval)

all_tags = [tag for tags in raw_recipes['tags_parsed'] for tag in tags]
from collections import Counter
tag_counts = Counter(all_tags)

print(f"\n=== Tags ===")
print(f"Total unique tags: {len(tag_counts):,}")
print(f"Avg tags per recipe: {raw_recipes['tags_parsed'].apply(len).mean():.1f}")
print(f"\nTop 20 most common tags:")
for tag, count in tag_counts.most_common(20):
    print(f"  {tag}: {count:,}")

In [ ]:
raw_recipes.columns

In [ ]:
user_stats.columns

In [ ]:
# Merge interaction data with user types and recipe data
interactions_with_types = train.merge(
    user_stats[['user_id', 'user_type']], 
    on='user_id', 
    how='left'
).merge(
    raw_recipes[['id', 'minutes', 'n_steps', 'n_ingredients']], 
    left_on='recipe_id',
    right_on='id',
    how='left'
)

# Plot minutes distribution by user type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution of Recipe Cooking Time by User Type', fontsize=13)

colors = {'cold': 'coral', 'warm': 'steelblue', 'hot': 'mediumseagreen'}

for ax, user_type in zip(axes, ['cold', 'warm', 'hot']):
    subset = interactions_with_types[interactions_with_types['user_type'] == user_type]
    sns.histplot(np.log1p(subset['minutes']), bins=40, kde=True, 
                 ax=ax, color=colors[user_type])
    ax.set_title(f"{user_type.capitalize()} Users (n={len(subset):,})")
    ax.set_xlabel('log(1 + Minutes)')
    ax.set_ylabel('Number of Interactions')
    ax.axvline(np.log1p(subset['minutes'].median()), color='black', 
               linestyle='--', label=f"Median: {subset['minutes'].median():.0f} mins")
    ax.legend()

plt.tight_layout()
plt.show()

# Summary stats
print(interactions_with_types.groupby('user_type')['minutes'].agg([
    'median', 'mean', 
    lambda x: np.percentile(x, 25),
    lambda x: np.percentile(x, 75)
]).round(1).rename(columns={'<lambda_0>': 'Q1', '<lambda_1>': 'Q3'}))

In [ ]:
print(interactions_with_types.groupby('user_type')[['n_steps', 'n_ingredients']].agg([
    'median',
    lambda x: np.percentile(x, 25),
    lambda x: np.percentile(x, 75)
]).round(1))

In [ ]:
print(f"Total unique tags: {len(tag_counts):,}")
print(f"Avg tags per recipe: {raw_recipes['tags_parsed'].apply(len).mean():.1f}")
print(f"\nTop 20 most common tags:")
for tag, count in tag_counts.most_common(20):
    print(f"  {tag}: {count:,}")

In [ ]:
# Tags to filter out - category headers and redundant metadata
meta_tags = {
    'preparation', 'time-to-make', 'course', 'main-ingredient', 
    'dietary', 'occasion', 'cuisine', 'equipment', 'taste-mood',
    'number-of-servings', 'low-in-something', 'meat'
}

# Separate meaningful tags from meta tags
meaningful_tag_counts = {k: v for k, v in tag_counts.items() 
                         if k not in meta_tags}
meta_tag_counts = {k: v for k, v in tag_counts.items() 
                   if k in meta_tags}

print(f"Meta/category tags removed: {len(meta_tag_counts):,}")
print(f"Meaningful tags remaining: {len(meaningful_tag_counts):,}")

# Distribution of meaningful tag frequencies
meaningful_counts = np.array(list(meaningful_tag_counts.values()))
print(f"\nMeaningful tag frequency distribution:")
print(f"  Tags appearing > 10,000 times: {(meaningful_counts > 10000).sum():,}")
print(f"  Tags appearing 1,000 - 10,000 times: {((meaningful_counts >= 1000) & (meaningful_counts <= 10000)).sum():,}")
print(f"  Tags appearing 100 - 999 times: {((meaningful_counts >= 100) & (meaningful_counts < 1000)).sum():,}")
print(f"  Tags appearing < 100 times: {(meaningful_counts < 100).sum():,}")

print(f"\nTop 30 meaningful tags:")
from collections import Counter
meaningful_counter = Counter(meaningful_tag_counts)
for tag, count in meaningful_counter.most_common(30):
    print(f"  {tag}: {count:,}")

In [ ]:
# Step 1: Define frequency threshold for tag inclusion
MIN_TAG_FREQUENCY = 100  # drops the 140 rare tags
selected_tags = [tag for tag, count in meaningful_counter.items() 
                 if count >= MIN_TAG_FREQUENCY]
print(f"Tags retained after frequency filter: {len(selected_tags)}")

# Step 2: Build multi-hot encoding for selected tags
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=selected_tags)
tag_matrix = mlb.fit_transform(raw_recipes['tags_parsed'].apply(
    lambda tags: [t for t in tags if t in set(selected_tags)]
))

print(f"Tag matrix shape: {tag_matrix.shape}")
print(f"Average tags per recipe after filtering: {tag_matrix.sum(axis=1).mean():.1f}")
print(f"Sparsity: {1 - tag_matrix.sum() / (tag_matrix.shape[0] * tag_matrix.shape[1]):.3f}")

# Step 3: Check how many recipes lose all tags after filtering
recipes_with_no_tags = (tag_matrix.sum(axis=1) == 0).sum()
print(f"Recipes with no tags after filtering: {recipes_with_no_tags:,}")

In [ ]:
# Merge interactions with user types and tag data
interactions_with_tags = train.merge(
    user_stats[['user_id', 'user_type']], 
    on='user_id', 
    how='left'
).merge(
    raw_recipes[['id', 'tags_parsed']], 
    left_on='recipe_id', 
    right_on='id',
    how='left'
)

# Explode tags so each row is one interaction-tag pair
interactions_exploded = interactions_with_tags.explode('tags_parsed')
interactions_exploded = interactions_exploded[
    interactions_exploded['tags_parsed'].isin(set(selected_tags))
]

# Get top 20 tags per user type
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle('Top 20 Tags by User Type', fontsize=13)

colors = {'cold': 'coral', 'warm': 'steelblue', 'hot': 'mediumseagreen'}

for ax, user_type in zip(axes, ['cold', 'warm', 'hot']):
    subset = interactions_exploded[interactions_exploded['user_type'] == user_type]
    top_tags = subset['tags_parsed'].value_counts().head(20)
    
    # Normalise by number of interactions for that user type
    n_interactions = len(interactions_with_tags[
        interactions_with_tags['user_type'] == user_type
    ])
    top_tags_pct = (top_tags / n_interactions * 100).round(2)
    
    top_tags_pct.sort_values().plot(kind='barh', ax=ax, color=colors[user_type])
    ax.set_title(f"{user_type.capitalize()} Users")
    ax.set_xlabel('% of Interactions')
    ax.set_ylabel('Tag')

plt.tight_layout()
plt.show()

# Print the actual numbers for comparison across user types
print("Top 15 tags by user type (% of interactions):\n")
for user_type in ['cold', 'warm', 'hot']:
    subset = interactions_exploded[interactions_exploded['user_type'] == user_type]
    n_interactions = len(interactions_with_tags[interactions_with_tags['user_type'] == user_type])
    top_tags = (subset['tags_parsed'].value_counts().head(15) / n_interactions * 100).round(2)
    print(f"{user_type.upper()}:")
    print(top_tags.to_string())
    print()

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Fit SVD and check how many components explain 80%, 90%, 95% of variance
svd_analysis = TruncatedSVD(n_components=100, random_state=42)
svd_analysis.fit(tag_matrix)

cumulative_variance = np.cumsum(svd_analysis.explained_variance_ratio_)

for threshold in [0.80, 0.90, 0.95]:
    n_components = np.argmax(cumulative_variance >= threshold) + 1
    print(f"Components needed for {threshold*100:.0f}% variance: {n_components}")

# Plot explained variance
plt.figure(figsize=(10, 5))
plt.plot(range(1, 101), cumulative_variance, marker='.', markersize=3)
plt.axhline(y=0.90, color='red', linestyle='--', label='90% variance')
plt.axhline(y=0.95, color='orange', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('SVD Explained Variance - Tag Matrix')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Individual explained variance (first 10 components):")
for i, var in enumerate(svd_analysis.explained_variance_ratio_[:10], 1):
    print(f"  Component {i}: {var:.4f} (cumulative: {cumulative_variance[i-1]:.4f})")

# Check what the first component looks like
first_component = svd_analysis.components_[0]
top_tags_component1 = sorted(zip(selected_tags, first_component), 
                              key=lambda x: abs(x[1]), reverse=True)[:15]
print("\nTop tags driving Component 1:")
for tag, weight in top_tags_component1:
    print(f"  {tag}: {weight:.4f}")

# Check if the issue is the matrix needs normalising first
from sklearn.preprocessing import normalize
tag_matrix_norm = normalize(tag_matrix, norm='l2')

svd_norm = TruncatedSVD(n_components=100, random_state=42)
svd_norm.fit(tag_matrix_norm)
cumulative_variance_norm = np.cumsum(svd_norm.explained_variance_ratio_)

print("\nAfter L2 normalisation:")
for threshold in [0.80, 0.90, 0.95]:
    n_components = np.argmax(cumulative_variance_norm >= threshold) + 1
    print(f"  Components needed for {threshold*100:.0f}% variance: {n_components}")

In [ ]:
# Extend to 200 components to see the full picture
svd_extended = TruncatedSVD(n_components=200, random_state=42)
svd_extended.fit(tag_matrix_norm)
cumulative_variance_ext = np.cumsum(svd_extended.explained_variance_ratio_)

print("Cumulative variance at key checkpoints (normalised matrix):")
for n in [50, 100, 150, 200]:
    print(f"  {n} components: {cumulative_variance_ext[n-1]:.3f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, 201), cumulative_variance_ext, marker='.', markersize=2)
plt.axhline(y=0.80, color='red', linestyle='--', label='80% variance')
plt.axhline(y=0.70, color='orange', linestyle='--', label='70% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('SVD Explained Variance - Tag Matrix (L2 Normalised)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calculate tag rates per user type
tag_rates = {}
for user_type in ['cold', 'warm', 'hot']:
    subset = interactions_exploded[interactions_exploded['user_type'] == user_type]
    n_interactions = len(interactions_with_tags[interactions_with_tags['user_type'] == user_type])
    tag_rates[user_type] = subset['tags_parsed'].value_counts() / n_interactions * 100

tag_rates_df = pd.DataFrame(tag_rates).fillna(0)

# Calculate variance across user types for each tag
tag_rates_df['variance'] = tag_rates_df[['cold', 'warm', 'hot']].var(axis=1)
tag_rates_df['max_diff'] = (tag_rates_df[['cold', 'warm', 'hot']].max(axis=1) - 
                             tag_rates_df[['cold', 'warm', 'hot']].min(axis=1))

print("Tags with lowest variance (most uniform across user types):")
print(tag_rates_df.nsmallest(20, 'variance')[['cold', 'warm', 'hot', 'variance', 'max_diff']].round(3))

print("\nTags with highest variance (most discriminative across user types):")
print(tag_rates_df.nlargest(20, 'variance')[['cold', 'warm', 'hot', 'variance', 'max_diff']].round(3))

# Define threshold for dropping uniform tags
VARIANCE_THRESHOLD = 0.5  # adjust after seeing the distribution
discriminative_tags = tag_rates_df[tag_rates_df['variance'] >= VARIANCE_THRESHOLD].index.tolist()
print(f"\nTags retained after variance filter: {len(discriminative_tags)}")
print(f"Tags dropped as non-discriminative: {len(tag_rates_df) - len(discriminative_tags)}")

In [ ]:
# Tier 1: discriminative tags - useful for user preference modelling
discriminative_tags = tag_rates_df[tag_rates_df['variance'] >= 0.5].index.tolist()

# Tier 2: content tags - useful for recipe-to-recipe similarity
# Keep tags that appear in at least 1% of recipes regardless of variance
recipe_tag_freq = pd.Series(
    {tag: (tag_matrix[:, i].sum() / len(recipes_df) * 100) 
     for i, tag in enumerate(selected_tags)}
)
content_tags = recipe_tag_freq[recipe_tag_freq >= 1.0].index.tolist()

# Combined set
all_useful_tags = list(set(discriminative_tags + content_tags))

print(f"Discriminative tags (variance >= 0.5): {len(discriminative_tags)}")
print(f"Content tags (appear in >= 1% of recipes): {len(content_tags)}")
print(f"Combined unique tags: {len(all_useful_tags)}")
print(f"\nTags in content but not discriminative (pure content signal):")
pure_content = [t for t in content_tags if t not in discriminative_tags]
print(pure_content)

In [ ]:
# Rebuild tag matrix with 185 combined tags
mlb_final = MultiLabelBinarizer(classes=all_useful_tags)
tag_matrix_final = mlb_final.fit_transform(
    raw_recipes['tags_parsed'].apply(
        lambda tags: [t for t in tags if t in set(all_useful_tags)]
    )
)

print(f"Final tag matrix shape: {tag_matrix_final.shape}")
print(f"Sparsity: {1 - tag_matrix_final.sum() / (tag_matrix_final.shape[0] * tag_matrix_final.shape[1]):.3f}")
print(f"Average tags per recipe: {tag_matrix_final.sum(axis=1).mean():.1f}")

# Normalise and run SVD
from sklearn.preprocessing import normalize
tag_matrix_norm = normalize(tag_matrix_final, norm='l2')

svd_final = TruncatedSVD(n_components=150, random_state=42)
svd_final.fit(tag_matrix_norm)
cumulative_variance_final = np.cumsum(svd_final.explained_variance_ratio_)

print("\nCumulative variance at key checkpoints:")
for n in [25, 50, 75, 100, 125, 150]:
    print(f"  {n} components: {cumulative_variance_final[n-1]:.3f}")

In [ ]:
# Final SVD transform with 100 components
svd = TruncatedSVD(n_components=100, random_state=42)
tag_embeddings = svd.fit_transform(tag_matrix_norm)

print(f"Tag embeddings shape: {tag_embeddings.shape}")
print(f"Variance captured: {svd.explained_variance_ratio_.sum():.3f}")

# Sanity check - verify embeddings look reasonable
print(f"\nEmbedding value range: [{tag_embeddings.min():.3f}, {tag_embeddings.max():.3f}]")
print(f"Mean embedding norm: {np.linalg.norm(tag_embeddings, axis=1).mean():.3f}")

# Store against recipe IDs for later merging
tag_embedding_df = pd.DataFrame(
    tag_embeddings,
    index=raw_recipes['id'],
    columns=[f'tag_svd_{i}' for i in range(100)]
)
print(f"\ntag_embedding_df shape: {tag_embedding_df.shape}")
print("Sample:")
print(tag_embedding_df.iloc[:3, :5])

In [ ]:
tag_embedding_df.index.name = 'id'

In [ ]:
print(raw_recipes.columns.tolist())

In [ ]:
print(type(raw_recipes['nutrition'].iloc[0]))
print(raw_recipes['nutrition'].head())

In [ ]:
import ast

# Parse the string into actual lists
raw_recipes['nutrition_parsed'] = raw_recipes['nutrition'].apply(ast.literal_eval)

# Expand into named columns
nutrition_cols = ['calories', 'total_fat_pdv', 'sugar_pdv', 'sodium_pdv', 
                  'protein_pdv', 'sat_fat_pdv', 'carbs_pdv']

raw_recipes[nutrition_cols] = pd.DataFrame(
    raw_recipes['nutrition_parsed'].tolist(), 
    index=raw_recipes.index
)

print(raw_recipes[nutrition_cols].head())
print("\n")
print(raw_recipes[nutrition_cols].describe())

In [ ]:
# Find a simple, well-known recipe to cross-reference
print(raw_recipes[['name', 'nutrition']].head(20))

In [ ]:
nutrition_cols = ['calories', 'total_fat_pdv', 'sugar_pdv', 'sodium_pdv', 
                  'protein_pdv', 'sat_fat_pdv', 'carbs_pdv']

raw_recipes[nutrition_cols] = pd.DataFrame(
    raw_recipes['nutrition_parsed'].tolist(), 
    index=raw_recipes.index
)

print(raw_recipes[nutrition_cols].head())
print("\n")
print(raw_recipes[nutrition_cols].describe())

In [ ]:
# Check 99th percentile values
print("99th percentile values:")
print(raw_recipes[nutrition_cols].quantile(0.99))

print("\n99.9th percentile values:")
print(raw_recipes[nutrition_cols].quantile(0.999))

# Check how many recipes would be affected at each threshold
for threshold in [0.99, 0.999]:
    mask = (raw_recipes[nutrition_cols] > raw_recipes[nutrition_cols].quantile(threshold)).any(axis=1)
    print(f"\nRecipes with any value above {threshold*100}th percentile: {mask.sum()} ({mask.sum()/len(raw_recipes)*100:.2f}%)")

In [ ]:
# Cap at 99th percentile
percentile_99 = raw_recipes[nutrition_cols].quantile(0.99)

for col in nutrition_cols:
    raw_recipes[col] = raw_recipes[col].clip(upper=percentile_99[col])

print("After capping at 99th percentile:")
print(raw_recipes[nutrition_cols].describe())

# Now scale using RobustScaler since distribution is still skewed
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
nutrition_scaled = scaler.fit_transform(raw_recipes[nutrition_cols])

nutrition_embedding_df = pd.DataFrame(
    nutrition_scaled,
    index=raw_recipes['id'],
    columns=[f'nutrition_{col}' for col in nutrition_cols]
)

print(f"\nNutrition embedding shape: {nutrition_embedding_df.shape}")
print(nutrition_embedding_df.describe().round(3))

In [ ]:
# Combine tag SVD embeddings and nutrition features
recipe_features = tag_embedding_df.join(nutrition_embedding_df, how='inner')

print(f"Recipe feature matrix shape: {recipe_features.shape}")
print(f"Expected: (231637, 107) — 100 tag SVD + 7 nutrition")
print(f"\nAny nulls: {recipe_features.isnull().sum().sum()}")
print(f"\nSample:")
print(recipe_features.head(3))

In [ ]:
import os

SAVE_DIR = 'data/processed'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the final recipe feature matrix
recipe_features.to_parquet(f'{SAVE_DIR}/recipe_features.parquet')

# Save the raw_recipes df with parsed columns (nutrition cols + tags_parsed)
raw_recipes.to_parquet(f'{SAVE_DIR}/raw_recipes_parsed.parquet')

# Save the SVD model and scaler so we can transform new recipes later
import joblib
joblib.dump(svd, f'{SAVE_DIR}/tag_svd_model.pkl')
joblib.dump(scaler, f'{SAVE_DIR}/nutrition_scaler.pkl')
joblib.dump(mlb_final, f'{SAVE_DIR}/tag_mlb.pkl')

print("Saved:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

In [ ]:
SAVE_DIR = '/kaggle/working/processed'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ----------------------------------------------------------------------
# Palette pulled from the slide
# ----------------------------------------------------------------------
NAVY      = "#1b2a3a"   # title
SLATE     = "#5b6b7b"   # subtitle / muted labels
GREEN     = "#3a6a1f"   # accent bar + section kicker
GREEN_BOX = "#e9f1dd"   # light highlight panel
INK       = "#1f1f1f"   # body
GRID      = "#e6e6e6"
FOOTER    = "#9a9a9a"

# ----------------------------------------------------------------------
# Global style
# ----------------------------------------------------------------------
sns.set_theme(style="white")
mpl.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Source Sans Pro", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.serif":        ["Source Serif Pro", "Georgia", "PT Serif", "DejaVu Serif"],
    "text.color":        INK,
    "axes.edgecolor":    "#c9c9c9",
    "axes.linewidth":    0.9,
    "axes.labelcolor":   SLATE,
    "xtick.color":       SLATE,
    "ytick.color":       SLATE,
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ----------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
fig.subplots_adjust(top=0.72, bottom=0.16, left=0.07, right=0.96, wspace=0.22)


# ----------------------------------------------------------------------
# Left: count of ratings  (green ramp instead of cubehelix)
# ----------------------------------------------------------------------
order = sorted(train["rating"].unique())
pal = sns.color_palette(f"light:{GREEN}", n_colors=len(order))
sns.countplot(x="rating", data=train, order=order, hue="rating", hue_order=order,
              palette=pal, legend=False, edgecolor="white", linewidth=0.6,
              ax=axes[0])
for c in axes[0].containers:
    axes[0].bar_label(c, fmt="%d", padding=3, color=SLATE, fontsize=9)
axes[0].set_title("Frequency of Each Rating", loc="left", color=NAVY,
                  fontsize=18, fontweight="semibold", pad=12)
axes[0].set(xlabel="Rating", ylabel="Count")
axes[0].grid(axis="x", visible=False)

# ----------------------------------------------------------------------
# Right: spread of ratings, themed in slide greens
# ----------------------------------------------------------------------
sns.boxplot(x="rating", data=train, color=GREEN_BOX, width=0.5, ax=axes[1],
            boxprops=dict(edgecolor=GREEN, linewidth=1.6),
            medianprops=dict(color=GREEN, linewidth=2.2),
            whiskerprops=dict(color=GREEN, linewidth=1.4),
            capprops=dict(color=GREEN, linewidth=1.4),
            flierprops=dict(marker="o", markerfacecolor=GREEN_BOX,
                            markeredgecolor=GREEN, markersize=4, alpha=0.6))
axes[1].set_title("Distribution of Ratings", loc="left", color=NAVY,
                  fontsize=18, fontweight="semibold", pad=12)
axes[1].set(xlabel="Rating", ylabel="")
axes[1].grid(axis="y", visible=False)


plt.show()
fig.savefig("rating_eda.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
train.shape

In [ ]:
import pandas as pd

# Load the pickle file
ingr_map = pd.read_pickle(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/ingr_map.pkl')

# Preview the first few rows
ingr_map.columns

In [ ]:
ingr_map.replaced.value_counts()

In [ ]:
recipe = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_recipes.csv')
recipe

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import seaborn as sns

# ----------------------------------------------------------------------
# Palette pulled from the slide (same as the rating-EDA figure)
# ----------------------------------------------------------------------
NAVY      = "#1b2a3a"
SLATE     = "#5b6b7b"
GREEN     = "#3a6a1f"
GREEN_BOX = "#e9f1dd"
INK       = "#1f1f1f"
GRID      = "#e6e6e6"
FOOTER    = "#9a9a9a"

sns.set_theme(style="white")
mpl.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Source Sans Pro", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.serif":        ["Source Serif Pro", "Georgia", "PT Serif", "DejaVu Serif"],
    "text.color":        INK,
    "axes.edgecolor":    "#c9c9c9",
    "axes.linewidth":    0.9,
    "axes.labelcolor":   SLATE,
    "xtick.color":       SLATE,
    "ytick.color":       SLATE,
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ----------------------------------------------------------------------
# Data (your logic, unchanged)
# ----------------------------------------------------------------------
item_rate_count = train.groupby("recipe_id")["user_id"].nunique().sort_values(ascending=False)
x_rank = range(len(item_rate_count))
count  = item_rate_count.value_counts()

# ----------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
fig.subplots_adjust(top=0.72, bottom=0.16, left=0.07, right=0.96, wspace=0.22)

# --- Editorial header ---
fig.add_artist(Rectangle((0.055, 0.80), 0.0045, 0.17, color=GREEN,
                         transform=fig.transFigure, clip_on=False))
fig.text(0.075, 0.94, "POPULARITY SKEW  ·  EDA",
         color=GREEN, fontsize=11, fontweight="bold")
fig.text(0.073, 0.855, "The long tail of item popularity",
         color=NAVY, fontsize=26, fontweight="bold", family="serif")
fig.text(0.075, 0.78, "A few recipes attract most ratings; the rest form a heavy tail.",
         color=SLATE, fontsize=12.5, fontstyle="italic")

# ----------------------------------------------------------------------
# Left: long tail
# ----------------------------------------------------------------------
axes[0].bar(x=x_rank, height=item_rate_count.values, width=1.0, align="edge",
            color=GREEN, edgecolor="none")
xr = np.arange(len(item_rate_count))
axes[0].fill_between(xr, item_rate_count.values, step="post", color=GREEN, alpha=0.22)
axes[0].plot(xr, item_rate_count.values, drawstyle="steps-post", color=GREEN, linewidth=0.8)
axes[0].set_xticks([])
axes[0].margins(x=0)
axes[0].set_xticks([])
axes[0].set_title("Long tail of rating frequency", loc="left", color=NAVY,
                  fontsize=14, fontweight="semibold", pad=12)
axes[0].set(xlabel="Items (ranked by popularity)", ylabel="# of ratings")
axes[0].grid(axis="x", visible=False)
axes[0].margins(x=0)

# ----------------------------------------------------------------------
# Right: log-log scatter
# ----------------------------------------------------------------------
sns.scatterplot(x=np.log1p(count.index), y=np.log1p(count.values), ax=axes[1],
                color=GREEN, edgecolor="white", linewidth=0.5, s=42, alpha=0.85)
axes[1].set_title("Log-log distribution", loc="left", color=NAVY,
                  fontsize=14, fontweight="semibold", pad=12)
axes[1].set(xlabel="# ratings (log1p scale)", ylabel="# items (log1p scale)")

# --- Footer rule ---
fig.add_artist(Line2D([0.07, 0.96], [0.06, 0.06], color="#e0e0e0",
                      linewidth=1, transform=fig.transFigure))
fig.text(0.07, 0.025, "Recommender EDA · train split", color=FOOTER, fontsize=9.5)

fig.savefig("longtail_eda.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import seaborn as sns

# ----------------------------------------------------------------------
# Palette pulled from the slide
# ----------------------------------------------------------------------
NAVY      = "#1b2a3a"
SLATE     = "#5b6b7b"
GREEN     = "#3a6a1f"
GREEN_BOX = "#e9f1dd"
INK       = "#1f1f1f"
GRID      = "#e6e6e6"
FOOTER    = "#9a9a9a"

sns.set_theme(style="white")
mpl.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Source Sans Pro", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.serif":        ["Source Serif Pro", "Georgia", "PT Serif", "DejaVu Serif"],
    "text.color":        INK,
    "axes.edgecolor":    "#c9c9c9",
    "axes.linewidth":    0.9,
    "axes.labelcolor":   SLATE,
    "xtick.color":       SLATE,
    "ytick.color":       SLATE,
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ----------------------------------------------------------------------
# Data (your logic, unchanged)
# ----------------------------------------------------------------------
item_counts = train.groupby('recipe_id')['rating'].count().sort_values(ascending=False)
freq_dist   = item_counts.value_counts().sort_index()

# ----------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.subplots_adjust(top=0.70, bottom=0.15, left=0.07, right=0.96, wspace=0.22)


# ----------------------------------------------------------------------
# Left: long tail
# ----------------------------------------------------------------------
x_rank = np.arange(len(item_counts))
axes[0].fill_between(x_rank, item_counts.values, step="post",
                     color=GREEN, alpha=0.22)
axes[0].plot(x_rank, item_counts.values, drawstyle="steps-post",
             color=GREEN, linewidth=0.9)
axes[0].set_xticks([])
axes[0].margins(x=0)
axes[0].set_title("Long tail of interactions", loc="left", color=NAVY,
                  fontsize=18, fontweight="semibold", pad=12)
axes[0].set(xlabel="Items (ranked by popularity)", ylabel="Number of ratings")
axes[0].grid(axis="x", visible=False)

# ----------------------------------------------------------------------
# Right: log-log distribution
# ----------------------------------------------------------------------
sns.scatterplot(x=freq_dist.index, y=freq_dist.values, ax=axes[1],
                color=GREEN, edgecolor="white", linewidth=0.5, s=42, alpha=0.85)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title("Log-log popularity distribution", loc="left", color=NAVY,
                  fontsize=18, fontweight="semibold", pad=12)
axes[1].set(xlabel="Number of ratings (log scale)",
            ylabel="Number of items (log scale)")
axes[1].grid(True, which="both", ls="--", color=GRID, alpha=0.7)

fig.savefig("longtail_eda.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
train.user_id.min()

In [ ]:
user = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_users.csv')
user.head(10)

In [ ]:
user.u.min()

In [ ]:
recipe = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/RAW_recipes.csv')
recipe.columns

In [ ]:
recipe.head()

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
interaction = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/interactions_train.csv')
interaction.columns

In [ ]:
user = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_users.csv')
user.columns

In [ ]:
recipe = pd.read_csv(r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/PP_recipes.csv')
recipe.columns

In [ ]:
import pandas as pd
import numpy as np

path = r'/kaggle/input/datasets/shuyangli94/food-com-recipes-and-user-interactions/'

train = pd.read_csv(path+'interactions_train.csv')
val = pd.read_csv(path+'interactions_validation.csv')
test = pd.read_csv(path+'interactions_test.csv')
pp_users = pd.read_csv(path+'PP_users.csv')
pp_recipes = pd.read_csv(path+'PP_recipes.csv')
raw_recipes = pd.read_csv(path+'RAW_recipes.csv')  # whatever the raw file is named

# Sanity checks
print("n_users (from pp_users):", len(pp_users))
print("n_items (from pp_recipes):", len(pp_recipes))
print("n_interactions train/val/test:", len(train), len(val), len(test))

# Verify u, i are contiguous 0..N-1
assert pp_users['u'].max() == len(pp_users) - 1
assert pp_recipes['i'].max() == len(pp_recipes) - 1

# Rating distribution
print(train['rating'].value_counts(normalize=True).sort_index())

In [ ]:
import ast
pp_recipes['ingredient_ids'] = pp_recipes['ingredient_ids'].apply(ast.literal_eval)
pp_users['items'] = pp_users['items'].apply(ast.literal_eval)
pp_users['ratings'] = pp_users['ratings'].apply(ast.literal_eval)

In [ ]:
# Join on the original recipe id
raw_recipes = raw_recipes.rename(columns={'id': 'recipe_id_raw'})
recipes = pp_recipes.merge(
    raw_recipes,
    left_on='id',
    right_on='recipe_id_raw',
    how='left'
)

# Now recipes has: i (index), ingredient_ids (list of int), tags (list of str),
# nutrition (calories/fat/sugar/protein/sodium/carbs), n_steps, minutes, etc.

In [ ]:
from scipy.sparse import csr_matrix

n_users = pp_users['u'].max() + 1
n_items = pp_recipes['i'].max() + 1

# Binarize: rating >= 4 counts as positive (or use all interactions)
train_pos = train[train['rating'] >= 4]

rows = train_pos['u'].values
cols = train_pos['i'].values
vals = np.ones(len(train_pos), dtype=np.float32)

user_item_matrix = csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))

from scipy.sparse import save_npz
save_npz('data/user_item_train.npz', user_item_matrix)

In [ ]:
train.rating.value_counts(normalize=True)

In [ ]:
user_stats['pct_five_star'] = train[train['rating'] == 5].groupby('user_id')['rating'].count() / user_stats['rating_count']